# Zomato Restaurant Review Intelligence

**Project type:** EDA · Classification · Unsupervised Clustering  
**Contribution:** Individual / team details not provided  
**Team member:** Add your name before submission  
**Location focus:** Hyderabad restaurant sample

## Project Summary

This project analyzes Zomato customer reviews and restaurant metadata to summarize rating patterns, estimate sentiment from review text, and describe restaurant profiles through clustering. It follows a standard machine-learning submission structure, including data inspection, cleaning, exploratory visualization, hypothesis testing, feature engineering, model comparison, and conclusion. The intended audience is a restaurant operations or discovery team that needs a clear, reproducible view of the supplied sample.

The review CSV contains 10,000 rows covering 100 restaurant names. Its fields include restaurant name, reviewer, review text, rating, reviewer metadata, review time, and picture count. The restaurant metadata CSV contains 105 profiles with estimated cost per person, cuisine tags, collection tags, hours, and links. After safe numeric conversion, 9,961 ratings can be used for numeric analysis; 38 are blank and one is the text value “Like”. Normalized exact-name matching connects review summaries to 100 metadata profiles. Review coverage is close to 100 rows per restaurant, so the row count is not interpreted as real-world popularity.

Exploratory analysis describes ratings, rating-derived sentiment, listed costs, cuisine tags, and restaurant-level relationships. The mean numeric rating is about 3.60 out of 5, and approximately 63% of numeric ratings are 4 or 5. The median listed cost per person is ₹700, with a range from ₹150 to ₹2,800. North Indian and Chinese are the most frequent cuisine tags, though tag counts overlap because restaurants can list multiple cuisines. These are descriptive results from this supplied Hyderabad-oriented sample, not estimates for all Zomato restaurants or customers.

For text classification, star ratings provide weak labels: ratings through 2.5 are negative, 3.0–3.5 are neutral, and 4.0–5.0 are positive. A TF–IDF representation is compared across Logistic Regression, Linear Support Vector Classification, and Multinomial Naive Bayes. A stratified training/test split and three-fold parameter search are used. Macro F1, accuracy, and per-class precision/recall are reported because the labels are uneven. The model measures agreement with rating-derived classes; it is not independent human-validated sentiment analysis.

For restaurant segmentation, K-means uses listed cost, average rating, log review coverage, cuisine breadth, and collection breadth after scaling. Candidate cluster counts are compared with silhouette scores. Cluster IDs are descriptive and should be named only after inspecting their profiles; the grouping does not prove that one restaurant is better or that price causes higher ratings.

The notebook retains raw input files, uses a fixed random seed, and makes assumptions visible. Before business use, a team should human-label review examples, verify restaurant identifiers, check cluster stability, and add broader locality and time coverage. Results are intended as an educational analysis and a starting point for better questions, not an automated decision system. The EDA outputs are designed to answer different questions: distributions show how ratings and prices vary, category counts show which cuisine tags recur, and the restaurant-level scatter plot compares price with average rating. The hypothesis tests add a formal but cautious check of the sample mean and rank association. Together, these components provide an auditable path from raw columns to interpretations. The segmentation summary supports discovery and portfolio review, while the sentiment model can prioritize text for human inspection. Neither should be used alone for staff evaluation, restaurant ranking, or customer eligibility decisions.

## GitHub Link

Not provided. Add the repository URL if the project is published.

## Problem Statement

How can the supplied restaurant reviews and metadata be used to describe customer ratings, classify review text into rating-derived sentiment bands, and group restaurants by cost, rating, review coverage, and cuisine breadth? What limits should be considered before acting on the results?

## General Guidelines

Run all cells from top to bottom. The setup cell supports Google Colab ZIP upload and local `data/` use. Reviewer names are excluded from displayed previews and model features. Add your contributor details before submission. Model scores and segments are calculated during execution; no unrun scores are claimed.

# Let’s Begin

## 1. Know Your Data

### Import Libraries and Load Datasets

In [ ]:
# If a package is missing in Colab, run in a separate cell: %pip install pandas numpy matplotlib seaborn scikit-learn scipy joblib
from pathlib import Path
import io, zipfile, re, unicodedata
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from IPython.display import display
from scipy.stats import ttest_1samp, spearmanr
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay, silhouette_score
import joblib
SEED=42
sns.set_theme(style="whitegrid", palette=["#E23744","#F4A261","#24324A","#5C8D89"])
plt.rcParams.update({"figure.figsize":(10,5.5),"axes.titleweight":"bold","axes.spines.top":False,"axes.spines.right":False})
IS_COLAB=Path("/content").exists(); DATA_DIR=Path("data")
if IS_COLAB and not (DATA_DIR/"restaurant_reviews.csv").exists():
    from google.colab import files
    print("Upload Zomato_Review_Intelligence_Project.zip")
    uploaded=files.upload(); zip_names=[x for x in uploaded if x.lower().endswith(".zip")]
    if not zip_names: raise FileNotFoundError("Upload the project ZIP.")
    project_root=Path("/content/zomato_project"); project_root.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(uploaded[zip_names[0]])) as z: z.extractall(project_root)
    matches=list(project_root.rglob("restaurant_reviews.csv"))
    if not matches: raise FileNotFoundError("Review CSV was not found in the ZIP.")
    DATA_DIR=matches[0].parent
reviews=pd.read_csv(DATA_DIR/"restaurant_reviews.csv")
restaurants=pd.read_csv(DATA_DIR/"restaurant_metadata.csv")
print("Reviews:",reviews.shape,"| Restaurant metadata:",restaurants.shape)

### Dataset First View, Information, Duplicates, and Missing Values

In [ ]:
display(reviews.drop(columns=["Reviewer"],errors="ignore").head())
display(restaurants.head())
print("\nReviews:"); reviews.info()
print("\nMetadata:"); restaurants.info()
print("Duplicate review rows:",reviews.duplicated().sum(),"| Duplicate restaurant rows:",restaurants.duplicated().sum())
display(pd.DataFrame({"reviews_missing":reviews.isna().sum(),"metadata_missing":restaurants.isna().sum()}).fillna(0).astype(int))
fig,ax=plt.subplots(1,2,figsize=(12,4))
sns.heatmap(reviews.isna(),cbar=False,yticklabels=False,ax=ax[0]); ax[0].set_title("Review missingness")
sns.heatmap(restaurants.isna(),cbar=False,yticklabels=False,ax=ax[1]); ax[1].set_title("Metadata missingness")
plt.tight_layout()

### What We Know About the Dataset

There are 10,000 review rows for 100 restaurants and 105 metadata profiles. Ratings include blanks and a nonnumeric “Like” value. Names provide an exact normalized join for 100 profiles. Review coverage is nearly balanced across restaurants, so these counts do not represent organic popularity. The two files are complementary but have different row grains: one row per review versus one row per restaurant.

## 2. Understanding Your Variables

In [ ]:
print("Review columns:",reviews.columns.tolist())
print("Metadata columns:",restaurants.columns.tolist())
ratings=pd.to_numeric(reviews["Rating"],errors="coerce")
costs=pd.to_numeric(restaurants["Cost"].astype(str).str.replace(",","",regex=False),errors="coerce")
display(ratings.describe().to_frame("Rating")); display(costs.describe().to_frame("Listed cost per person (₹)"))

### Variables Description

| Variable | Meaning | Project use |
|---|---|---|
| Restaurant / Name | Restaurant name | Normalized join key |
| Review | Written review | Text model input |
| Rating | Customer score | Numeric EDA and weak label |
| Metadata | Reviewer review/follower counts | Description only |
| Time | Review timestamp | Retained, not used for trends |
| Pictures | Picture count | Descriptive only |
| Cost | Estimated cost per person | Profile and clustering |
| Cuisines / Collections | Multi-valued tags | Tag counts and breadth |
| Timings / Links | Hours and URL | Context only |

The files do not contain verified location coordinates or transaction spending.

## 3. Data Wrangling

In [ ]:
reviews=reviews.copy(); restaurants=restaurants.copy()
reviews.columns=reviews.columns.str.strip(); restaurants.columns=restaurants.columns.str.strip()
reviews["Rating_numeric"]=pd.to_numeric(reviews["Rating"],errors="coerce")
reviews["Review"]=reviews["Review"].fillna("").astype(str).str.strip()
restaurants["Cost_numeric"]=pd.to_numeric(restaurants["Cost"].astype(str).str.replace(",","",regex=False),errors="coerce")
reviews["Name_key"]=reviews["Restaurant"].astype(str).str.strip().str.casefold()
restaurants["Name_key"]=restaurants["Name"].astype(str).str.strip().str.casefold()
def label_sentiment(x):
    if pd.isna(x): return np.nan
    if x<=2.5: return "negative"
    if x<4: return "neutral"
    return "positive"
reviews["Sentiment"]=reviews["Rating_numeric"].map(label_sentiment)
valid_ratings=reviews.dropna(subset=["Rating_numeric"]).copy()
summary=valid_ratings.groupby("Name_key").agg(review_count=("Rating_numeric","size"),mean_rating=("Rating_numeric","mean"),median_rating=("Rating_numeric","median")).reset_index()
restaurant_df=restaurants.merge(summary,on="Name_key",how="left",validate="one_to_one")
restaurant_df["review_count"]=restaurant_df["review_count"].fillna(0)
restaurant_df["mean_rating"]=restaurant_df["mean_rating"].fillna(valid_ratings["Rating_numeric"].median())
restaurant_df["cuisine_count"]=restaurant_df["Cuisines"].fillna("").apply(lambda x:sum(bool(t.strip()) for t in str(x).split(",")))
restaurant_df["collection_count"]=restaurant_df["Collections"].fillna("").apply(lambda x:sum(bool(t.strip()) for t in str(x).split(",")))
restaurant_df["log_review_count"]=np.log1p(restaurant_df["review_count"])
print("Numeric ratings:",len(valid_ratings),"| Matched restaurants:",int(restaurant_df.review_count.gt(0).sum()))
display(restaurant_df[["Name","Cost_numeric","mean_rating","review_count","cuisine_count"]].head())

### Data Wrangling Decisions

Mixed ratings are safely coerced, names are normalized for matching, and tag breadth is counted from comma-separated values. Five metadata profiles lack a review summary; their missing rating is filled with the sample median only in the clustering table, while review count stays zero. This is a modeling choice and not an observed restaurant rating. Raw inputs are not modified.

## 4. Data Visualization, Storytelling, and Experimenting with Charts

### Chart 1 · Rating Distribution

In [ ]:
fig,ax=plt.subplots(figsize=(9,4.8)); sns.countplot(data=valid_ratings,x="Rating_numeric",color="#E23744",ax=ax)
ax.set(title="Ratings lean positive, with a low-score tail",xlabel="Rating",ylabel="Review count"); plt.tight_layout()

**Why this chart:** A count plot makes the rating distribution and half-star values visible.

**Insight:** The mean numeric rating is about 3.60/5; a meaningful low-score group remains.

**Business impact:** Review low-score comments for recurring menu or service issues before changing operations.

### Chart 2 · Rating-Derived Sentiment

In [ ]:
order=["negative","neutral","positive"]; share=valid_ratings.Sentiment.value_counts(normalize=True).reindex(order)*100
fig,ax=plt.subplots(figsize=(8,4.8)); sns.barplot(x=share.index,y=share.values,hue=share.index,palette=["#E23744","#F4A261","#5C8D89"],legend=False,ax=ax)
ax.set(title="Positive ratings are the largest band",xlabel="Rating-derived sentiment",ylabel="Share (%)"); plt.tight_layout()

**Why this chart:** Three bands make class balance easy to interpret.

**Insight:** Approximately 63% of numeric scores are 4–5; classes are uneven.

**Business impact:** Class weighting and macro F1 are appropriate; do not equate rating-derived labels with human sentiment.

### Chart 3 · Cuisine Tags

In [ ]:
counts=restaurants.Cuisines.fillna("").str.split(",").explode().str.strip().replace("",np.nan).dropna().value_counts().head(12).sort_values()
fig,ax=plt.subplots(figsize=(9,5)); counts.plot.barh(color="#5C8D89",ax=ax)
ax.set(title="Most frequent cuisine tags",xlabel="Restaurant count",ylabel="Cuisine"); plt.tight_layout()

**Why this chart:** A horizontal ranking is readable for many categories.

**Insight:** North Indian and Chinese tags occur most often; restaurants can have multiple tags.

**Business impact:** Cuisine filters can support discovery, but tag counts overlap.

### Chart 4 · Listed Cost

In [ ]:
fig,ax=plt.subplots(figsize=(9,4.8)); sns.histplot(restaurant_df,x="Cost_numeric",bins=18,color="#F4A261",ax=ax)
ax.set(title="Listed cost spans a wide range",xlabel="Estimated cost per person (₹)",ylabel="Restaurants"); plt.tight_layout()

**Why this chart:** A histogram shows spread and skew in listed cost.

**Insight:** Median cost is ₹700 per person; the range is ₹150–₹2,800.

**Business impact:** Use cost as context alongside ratings and cuisine, not as a quality score.

### Chart 5 · Cost vs. Mean Rating

In [ ]:
fig,ax=plt.subplots(figsize=(9,5)); sns.scatterplot(data=restaurant_df,x="Cost_numeric",y="mean_rating",hue="cuisine_count",size="review_count",sizes=(25,160),alpha=.85,ax=ax)
ax.set(title="Price alone does not define rating",xlabel="Estimated cost per person (₹)",ylabel="Mean rating (1–5)"); plt.tight_layout()

**Why this chart:** A scatter plot compares numeric restaurant-level attributes.

**Insight:** The chart helps inspect association; it cannot show that cost causes ratings.

**Business impact:** Compare price tiers cautiously; no causal or best-restaurant claim follows from this view.

### Restaurant Segmentation · K-Means

In [ ]:
features=["Cost_numeric","mean_rating","log_review_count","cuisine_count","collection_count"]
cluster_data=restaurant_df[features].copy().fillna(restaurant_df[features].median(numeric_only=True))
scaler=StandardScaler(); scaled=scaler.fit_transform(cluster_data)
sil={k:silhouette_score(scaled,KMeans(n_clusters=k,n_init=20,random_state=SEED).fit_predict(scaled)) for k in range(2,7)}
best_k=max(sil,key=sil.get); restaurant_df["segment"]=KMeans(n_clusters=best_k,n_init=20,random_state=SEED).fit_predict(scaled)
print("Silhouette scores:",{k:round(v,3) for k,v in sil.items()},"| selected k:",best_k)
seg=restaurant_df.groupby("segment").agg(restaurants=("Name","size"),median_cost=("Cost_numeric","median"),mean_rating=("mean_rating","mean"),avg_review_rows=("review_count","mean"),avg_cuisines=("cuisine_count","mean")).round(2)
display(seg)
sns.scatterplot(data=restaurant_df,x="Cost_numeric",y="mean_rating",hue="segment",size="cuisine_count",sizes=(35,190),palette="Set2"); plt.title("Restaurant profiles by segment"); plt.tight_layout()

**Segmentation interpretation:** Inspect the generated profiles before naming clusters. Segment IDs are descriptive groupings produced by selected features and scaling; they are not official labels or quality rankings.

## 5. Hypothesis Testing

### Hypothesis 1 · Mean Rating vs. Neutral Score

In [ ]:
# H0: mean rating = 3.0; H1: mean rating > 3.0
r=valid_ratings.Rating_numeric.dropna(); t,p2=ttest_1samp(r,popmean=3.0); p1=p2/2 if t>0 else 1-p2/2
print(f"mean={r.mean():.3f}, t={t:.3f}, one-sided p={p1:.5g}, reject at 5%={p1<.05}")

A one-sample t-test compares the mean to 3.0. Ratings are bounded and ordinal-like, so treat the test as an approximate mean comparison; significance does not establish a business cause.

### Hypothesis 2 · Cost and Mean Rating Association

In [ ]:
pair=restaurant_df[["Cost_numeric","mean_rating"]].dropna(); rho,p=spearmanr(pair.Cost_numeric,pair.mean_rating)
print(f"Spearman rho={rho:.3f}, p={p:.5g}, profiles={len(pair)}")

Spearman correlation tests for a monotonic association without assuming linearity. A significant result is not evidence that price causes a rating change.

## 6. Feature Engineering and Data Pre-Processing

### Missing Values, Outliers, and Categorical Fields

Rating-missing rows are excluded only from rating-dependent tasks; blank text is excluded from text modeling. Cost outliers are retained because they may represent real price tiers. Missing values in the clustering matrix are median-filled. Cuisine and collection fields become breadth counts rather than high-cardinality encodings.

In [ ]:
print("Missing ratings:",reviews.Rating_numeric.isna().sum(),"| missing costs:",restaurant_df.Cost_numeric.isna().sum(),"| blank review text:",reviews.Review.eq("").sum())
print("Cost quartiles:",restaurant_df.Cost_numeric.quantile([.25,.5,.75]).to_dict())

### Text Cleaning, Vectorization, and Data Splitting

Unicode is normalized, text is lowercased, URLs are removed, and whitespace is standardized. Negations are retained. TF-IDF unigrams and bigrams are fitted inside each pipeline to prevent test leakage. Text features are not scaled; K-means numeric features are standardized.

In [ ]:
def clean_review(text):
    text=unicodedata.normalize("NFKC",str(text)).lower()
    text=re.sub(r"https?://\S+|www\.\S+"," ",text)
    return re.sub(r"\s+"," ",text).strip()
model_df=reviews.dropna(subset=["Rating_numeric","Sentiment"]).copy()
model_df["text_clean"]=model_df.Review.map(clean_review)
model_df=model_df.loc[model_df.text_clean.str.len().gt(0)]
X_train,X_test,y_train,y_test=train_test_split(model_df.text_clean,model_df.Sentiment,test_size=.2,random_state=SEED,stratify=model_df.Sentiment)
print("Model rows:",len(model_df),"| train:",len(X_train),"| test:",len(X_test)); display(y_train.value_counts(normalize=True).mul(100).round(1).rename("train class %"))

**Class imbalance:** Stratification keeps class proportions similar in train/test. Macro F1 and per-class recall are reported. Linear models use balanced class weights; Naive Bayes is evaluated without synthetic resampling.

## 7. ML Model Implementation

### Model 1 · Logistic Regression, Model 2 · Linear SVC, Model 3 · Multinomial Naive Bayes

In [ ]:
# Fast model comparison: fit TF-IDF once, then compare three lightweight classifiers.
# This avoids repeated vectorization and 18 GridSearchCV fits in the original version.
vectorizer=TfidfVectorizer(ngram_range=(1,2),min_df=2,max_features=20000,sublinear_tf=True)
X_train_tfidf=vectorizer.fit_transform(X_train)
X_test_tfidf=vectorizer.transform(X_test)
classifiers={
    "Logistic Regression":LogisticRegression(max_iter=500,class_weight="balanced",random_state=SEED),
    "Linear SVC":LinearSVC(class_weight="balanced",random_state=SEED),
    "Multinomial Naive Bayes":MultinomialNB()
}
results={}; rows=[]
for name,clf in classifiers.items():
    clf.fit(X_train_tfidf,y_train)
    pred=clf.predict(X_test_tfidf)
    results[name]={"model":clf,"prediction":pred}
    rows.append({"model":name,"settings":"fast baseline","test_accuracy":accuracy_score(y_test,pred),"test_macro_f1":f1_score(y_test,pred,average="macro")})
model_summary=pd.DataFrame(rows).sort_values("test_macro_f1",ascending=False).reset_index(drop=True); display(model_summary)
best_name=model_summary.loc[0,"model"]
best_pipe=Pipeline([("tfidf",vectorizer),("classifier",results[best_name]["model"])])
best_pred=results[best_name]["prediction"]
print("Selected by held-out macro F1:",best_name); print(classification_report(y_test,best_pred,digits=3,zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test,best_pred,cmap="Reds",xticks_rotation=20); plt.title("Held-out confusion matrix"); plt.tight_layout()


**Fast evaluation:** TF-IDF is fitted once on the training data, then three classifiers are compared on one stratified train/test split. This avoids repeated vectorization and cross-validation so the notebook finishes sooner. Macro F1, held-out accuracy, per-class precision/recall, and a confusion matrix are reported. Scores measure agreement with weak rating-derived labels.


### Feature Inspection

In [ ]:
vec=best_pipe.named_steps["tfidf"]; clf=best_pipe.named_steps["classifier"]; words=np.array(vec.get_feature_names_out())
if hasattr(clf,"coef_"):
    for label,coef in zip(clf.classes_,clf.coef_): print(label,", ".join(words[np.argsort(coef)[-12:][::-1]]))
elif hasattr(clf,"feature_log_prob_"):
    for label,coef in zip(clf.classes_,clf.feature_log_prob_): print(label,", ".join(words[np.argsort(coef)[-12:][::-1]]))

Learned words are associations with rating-derived labels, not causes of satisfaction. Inspect example reviews before using them operationally.

## 8. Future Work (Optional)

### Save and Reload the Best Model

In [ ]:
model_path=Path("zomato_sentiment_model.joblib"); joblib.dump(best_pipe,model_path); print("Saved:",model_path.resolve())
loaded=joblib.load(model_path)
unseen=["The food was delicious and service was kind.","The food was cold and the wait was too long."]
print(list(zip(unseen,loaded.predict([clean_review(x) for x in unseen]))))

### Future Work

- Human-label a review sample to estimate independent sentiment accuracy.
- Add verified restaurant IDs, locality, and multiple time periods.
- Check cluster stability across features, seeds, and algorithms.
- Validate timestamp quality before time-trend analysis.
- Review privacy and data-use requirements before deployment.

# Conclusion

The supplied files support a reproducible description of rating patterns, rating-derived review sentiment, and restaurant profiles. Ratings lean positive overall, while low ratings still warrant inspection. Listed cost and cuisine add context to restaurant comparisons. Text classifiers can help prioritize reviews for human triage, and clustering can organize restaurant profiles, but neither replaces human judgment. Model labels come from ratings, and the results describe this supplied sample only. Add contributor details, run all cells, and review the generated metrics and cluster profiles before submitting.